# Batch #8 - Frozen-encoder transfer: MiniLM vs e5-large

**Where this comes from.** Batch 7 probed the frozen representations on split-42. The
headline is narrow: `e5_large` comment-only AUC is **.8007 vs .7713 for MiniLM** (+2.94pt,
paired video-cluster bootstrap P=.964), and the gain is concentrated in the
`yes_implicit|yes` quadrant (.338 -> .562 accuracy). `mmbert_base` is flat/worse and
`bge_m3` is unstable once context is concatenated, so e5 is the only remaining candidate.

**Why both arms are frozen.** The batch-6 `A160r` controls fine-tune the top 3 MiniLM blocks.
Swapping in e5 and comparing *frozen* e5 to *fine-tuned* MiniLM would confound the backbone
with the freeze. This batch freezes both encoders (`--unfreeze-layers 0`), so the only
difference between the two arms is the representation. `A160r` remains the deployment
baseline and is quoted separately.

**Runs (6, ~1.5-2 h on T4 x2):**

| tag | backbone | encoder | what it measures |
|---|---|---|---|
| `MF_s42/43/44` | MiniLM | frozen | frozen MiniLM control; the cost of freezing |
| `E5F_s42/43/44` | `intfloat/multilingual-e5-large` | frozen | does the probe's representation gain transfer? |

Shared encoder, A160r recipe, same split-42, only the seed varies. Resume-safe.
The E5F arm uses batch 32 like the control; if a T4 OOMs, reduce only E5F to 16 and
record that change rather than silently mixing batch sizes.

**Decision rule.**
- `E5F > MF` by >= 1pt mean macro-F1 (or positive paired AUC) -> the representation gain is
  real; next test is e5 with the top 1-2 blocks re-opened, or a mixed e5-comment /
  MiniLM-video tower.
- `E5F ~ MF` -> freezing costs more than the probe's +2.9 AUC; close the encoder track.
- `E5F > A160r` (mean .7383, 3-seed ensemble .7543) -> immediate single-model candidate.

The probe's context variants are not used here: they concatenate fields into one mean-pooled
sequence, while this model keeps separate comment and video towers.


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = 'f7ee1cf1eccfd9bc4b92b902409152250b598af1'


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)


## 1. Data + GPU check

Attach `amnpham/lgbtqia` (Add Input), accelerator GPU T4 x2, Internet On. Builds the split-42
train/val frames and saves the val-side arrays the local analysis reads back
(labels, langs, videos, and the stereotype|hate quadrant per val row).


In [ ]:
import glob
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
df_all = data_mod.load_frame(languages=['EN', 'IT', 'NL'])
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)
labs = df_val['st_y'].astype(int).to_numpy()
assert len(labs) == 738 and int(labs.sum()) == 276, \
    f'unexpected val frame: n={len(labs)} pos={int(labs.sum())}'
np.save('weights/val_labels.npy', labs)
np.save('weights/val_langs.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos.npy', df_val['yt_title'].to_numpy(dtype=str))
quads = (df_val['stereotype'].astype(str) + '|' + df_val['hate_speech'].astype(str)).to_numpy(dtype=str)
np.save('weights/val_quads.npy', quads)
print('val:', len(df_val), 'rows | pos', int(labs.sum()), '| clusters', df_val['yt_title'].nunique(),
      flush=True)
print('  val quadrants', dict(pd.Series(quads).value_counts()), flush=True)
qtr = df_train['stereotype'].astype(str) + '|' + df_train['hate_speech'].astype(str)
print('  train quadrants', dict(qtr.value_counts()), flush=True)


## 2. Run list and helpers

A160 recipe unchanged (`dropout 0.3, wd 0.01, head-lr 1e-3, xattn-lr 1e-4, backbone-lr 2e-5,
unfreeze 3, layers 3, shared encoder, no pooled-v, pool-init 0.02, 160/256, 15 epochs, patience 5`).
The only change is `--context-dropout`. Everything runs at the same code pin as batch 5.


In [ ]:
BASE = dict(dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
            backbone_lr=2e-5, unfreeze_layers=0, layers=3, shared_encoder=True,
            pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
            max_len=160, video_max_len=256, no_amp=False)

RUNS = []
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'MF_s{s}', seed=s, split_seed=42,
                     backbone=model_xattn.DEFAULT_BACKBONE))
for s in (42, 43, 44):
    RUNS.append(dict(BASE, tag=f'E5F_s{s}', seed=s, split_seed=42,
                     backbone='intfloat/multilingual-e5-large'))

# Batch-6 same-split deployment baseline; not re-run here.
RECORDED = {'A160r_s42': 0.7476, 'A160r_s43': 0.7396, 'A160r_s44': 0.7276}
RECORDED_MEAN = 0.7383
RECORDED_X3 = 0.7543


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if os.path.exists(hist):
        print(f"-- skip (history exists): {hist}", flush=True)
        return True
    cmd = (
        'python src/train_xattn.py --languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        f"--backbone {r['backbone']} "
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"--context-dropout 0 "
        f"{'--pooled-v' if r['pooled_v'] else '--no-pooled-v'} "
        f"{'--shared-encoder ' if r['shared_encoder'] else ''}"
        f"{'--no-amp ' if r['no_amp'] else ''}"
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    best = float(h.loc[bi, 'val_macro_f1'])
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {best:.4f} | acc {h.loc[bi, 'val_acc']:.4f} | {len(h)} epochs",
          flush=True)
    return True


def dump_probs(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    if not os.path.exists(hist) or not os.path.exists(out):
        print(f'-- no probs for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        r['backbone'], unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
    ).to(dev)
    net.load_state_dict(torch.load(out, map_location=dev))
    net.eval()
    tok_r = load_tokenizer(r['backbone'])
    if tok_r.pad_token is None:
        tok_r.pad_token = tok_r.eos_token or tok_r.sep_token
    loader = tx.make_loader(df_val, tok_r, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=True)
    np.save(f"weights/{r['tag']}_val_probs.npy", o['probs'])
    m = tx.metrics_from(o['preds'], o['labels'])
    print(f"   {r['tag']}: reload-eval macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned:', len(RUNS), 'runs |', [r['tag'] for r in RUNS], flush=True)


## 3. Run everything (~4-6 h on T4 x2)

12 runs: 3 same-session controls + 3 dropout rates x 3 seeds. Resume-safe (history-exists
skip); partial results zipped after every run. The first six runs give the primary
control-vs-CD30 answer.


In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS, 1):
    print(f'=== [{i}/{len(RUNS)}] {r["tag"]} (elapsed {(time.time() - t0) / 60:.0f} min) ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        dump_probs(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! dump_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        zip_small()
    except Exception as e:                           # noqa: BLE001
        print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
print(f'all {len(RUNS)} runs finished in {(time.time() - t0) / 3600:.2f} h', flush=True)


## 4. In-notebook summary

History table + per-quadrant FN/FP + predicted positive rate for every run; 3-seed ensemble
per arm; same-session control mean. The rigorous paired statistics (cluster bootstrap over
the 56 val videos) run locally on the pulled arrays in `batch6_analysis.py`.


In [ ]:
rows = []
for r in RUNS:
    hist = f"weights/{r['tag']}_history.csv"
    if not os.path.exists(hist):
        continue
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    arm = r['tag'].split('_')[0]
    rows.append(dict(arm=arm, tag=r['tag'], seed=r['seed'], backbone=r['backbone'],
                     epochs=len(h), best_epoch=int(h.loc[bi, 'epoch']),
                     best_f1=float(h.loc[bi, 'val_macro_f1']),
                     best_acc=float(h.loc[bi, 'val_acc'])))
summ = pd.DataFrame(rows).sort_values(['arm', 'seed'])
summ.to_csv('weights/batch8_summary.csv', index=False)
print(summ.to_string(index=False), flush=True)

labs = np.load('weights/val_labels.npy')
quads = np.load('weights/val_quads.npy')


def quad_line(probs):
    pred = probs.argmax(1)
    out = []
    for q in ('yes|no', 'yes|yes_implicit', 'yes|yes_explicit',
              'no|no', 'no|yes_implicit', 'no|yes_explicit'):
        m = quads == q
        n = int(m.sum())
        if q.startswith('yes'):
            out.append(f'{q} FN {int((pred[m] == 0).sum()):>2}/{n:<3}')
        else:
            out.append(f'{q} FP {int((pred[m] == 1).sum()):>2}/{n:<3}')
    return ' | '.join(out)


probs_by_tag = {}
for r in RUNS:
    p = f"weights/{r['tag']}_val_probs.npy"
    if not os.path.exists(p):
        print(f'missing probs: {r["tag"]}', flush=True)
        continue
    probs_by_tag[r['tag']] = np.load(p)
    pr = probs_by_tag[r['tag']].argmax(1)
    m = tx.metrics_from(pr.tolist(), labs.tolist())
    print(f"{r['tag']:<12} macro_f1 {m['macro_f1']:.4f} acc {m['acc']:.4f} "
          f"pos {pr.mean():.3f} | {quad_line(probs_by_tag[r['tag']])}", flush=True)

print()
print('arm summary (mean best F1 over seeds; 3-seed ensemble):')
for arm in ('MF', 'E5F'):
    tags = [t for t in probs_by_tag if t.startswith(arm + '_s')]
    if not tags:
        continue
    best = [float(summ.loc[summ.tag == t, 'best_f1'].iloc[0]) for t in tags]
    pos = [float((probs_by_tag[t].argmax(1) == 1).mean()) for t in tags]
    line = (f'  {arm:<3} seeds {len(tags)} | mean best {np.mean(best):.4f} '
            f'| mean pos {np.mean(pos):.3f}')
    if len(tags) >= 2:
        ens = np.mean([probs_by_tag[t] for t in tags], axis=0)
        me = tx.metrics_from(ens.argmax(1).tolist(), labs.tolist())
        line += f' | ensemble {me["macro_f1"]:.4f}'
    print(line, flush=True)

print()
print(f'batch-6 A160r baseline: mean best {RECORDED_MEAN:.4f} | 3-seed ensemble {RECORDED_X3:.4f}')
if 'MF_s42' in probs_by_tag and 'E5F_s42' in probs_by_tag:
    mf = float(summ.loc[summ.arm == 'MF', 'best_f1'].mean())
    e5 = float(summ.loc[summ.arm == 'E5F', 'best_f1'].mean())
    print(f'  E5F - MF mean delta: {e5 - mf:+.4f}', flush=True)

zip_small()
print('BATCH8 COMPLETE', flush=True)
